# Memory AI Lab — Évaluation ARI V3

**Pré-requis :** activer le GPU dans Colab → `Exécution > Modifier le type d'exécution > GPU T4`

**Structure attendue dans Google Drive :**
```
Mon Drive/
  memory_ai_lab/
    src/          ← dossier source du projet
    data/
      group_anon.txt
      group_gold.json
```

In [ ]:
# ── CELLULE 1 : Monter Google Drive ───────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT = '/content/drive/MyDrive/memory_ai_lab'
assert os.path.exists(PROJECT), f"Dossier introuvable : {PROJECT}"
print(f"✓ Projet trouvé : {PROJECT}")

In [ ]:
# ── CELLULE 2 : Installer les dépendances ─────────────────────────────────
# (à run une seule fois par session Colab)
!pip install sentence-transformers scikit-learn -q
!pip install spacy -q
!python -m spacy download fr_core_news_sm -q
print("✓ Dépendances installées")

In [ ]:
# ── CELLULE 3 : Setup du path Python ──────────────────────────────────────
import sys
sys.path.insert(0, f'{PROJECT}/src')

# Vérif GPU
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"✓ Device : {device}")
if device == 'cuda':
    print(f"  GPU : {torch.cuda.get_device_name(0)}")

In [ ]:
# ── CELLULE 4 : Imports ────────────────────────────────────────────────────
import json
import numpy as np
from pathlib import Path
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sentence_transformers import SentenceTransformer

from parsers.whatsapp_parser import parse_whatsapp_chat
from episode_algorithm import EpisodeSegmenter
from episode_splitter import EpisodeSplitter, SplitConfig

DATA_DIR  = f'{PROJECT}/data'
CACHE_DIR = f'{PROJECT}/data'
print("✓ Imports OK")

In [ ]:
# ── CELLULE 5 : Embeddings avec GPU + cache sur Drive ─────────────────────
# Si group_embeddings.npy existe déjà → rechargé en <1 sec
# Sinon → calculé en ~10 sec avec GPU T4

EMBED_CACHE = Path(CACHE_DIR) / 'group_embeddings.npy'

print("[1/4] Parsing ...")
artifacts = parse_whatsapp_chat(f'{DATA_DIR}/group_anon.txt')
texts = [a.content for a in artifacts]
print(f"      {len(texts)} messages")

if EMBED_CACHE.exists():
    print("[2/4] Embeddings (cache) ...")
    embeddings = np.load(EMBED_CACHE)
    assert len(embeddings) == len(texts), "Cache périmé — supprimer group_embeddings.npy"
else:
    print(f"[2/4] Embeddings (calcul sur {device}) ...")
    model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=device)
    embeddings = model.encode(texts, batch_size=256, show_progress_bar=True,
                              device=device, convert_to_numpy=True).astype(np.float32)
    np.save(EMBED_CACHE, embeddings)
    print(f"      Sauvegardé → {EMBED_CACHE}")

print(f"      Shape : {embeddings.shape}")

In [ ]:
# ── CELLULE 6 : Chargement gold labels ────────────────────────────────────
print("[3/4] Gold labels ...")

with open(f'{DATA_DIR}/group_gold.json', encoding='utf-8') as f:
    gold = json.load(f)

n = len(artifacts)
y_true = [None] * n
for ep in gold['episodes']:
    for idx in range(ep['start_idx'], ep['end_idx'] + 1):
        if idx < n:
            y_true[idx] = ep['episode_id']

n_gold = sum(1 for l in y_true if l is not None)
n_ep_gold = len(set(l for l in y_true if l is not None))
print(f"      {n_gold} msgs labellisés, {n_ep_gold} épisodes gold")

In [ ]:
# ── CELLULE 7 : Segmentation V3 ───────────────────────────────────────────
# Modifie les paramètres ici pour experimenter
print("[4/4] Segmentation V3 ...")

SEG_KWARGS = dict(
    time_threshold_minutes=120,
    attach_threshold=0.30,
    alpha=0.45, beta=0.25, gamma=0.10, delta=0.20, rho=0.05,
    dormancy_minutes=1440,
    ema_alpha=0.80,
    active_penalty_hours=24.0,
    hard_break_minutes=720,
    allow_reactivation=True,
)

segmenter = EpisodeSegmenter(**SEG_KWARGS)
episodes  = segmenter.segment(artifacts, embeddings)
episodes  = segmenter.consolidate(episodes)

splitter = EpisodeSplitter(SplitConfig(
    min_cohesion=0.65, min_size_to_split=8,
    max_span_hours=168.0, max_splits=6,
    min_sub_size=3, silhouette_threshold=0.10,
))
episodes = splitter.split(episodes, artifacts, embeddings)

y_pred = [None] * n
for ep in episodes:
    for idx in ep.artifact_indices:
        if idx < n:
            y_pred[idx] = ep.id

n_ep_pred = len(episodes)
print(f"      {n_ep_pred} épisodes prédits")

In [ ]:
# ── CELLULE 8 : Résultats ──────────────────────────────────────────────────
pairs = [(t, p) for t, p in zip(y_true, y_pred) if t is not None and p is not None]
yt, yp = zip(*pairs)
ari = adjusted_rand_score(yt, yp)
nmi = normalized_mutual_info_score(yt, yp)

print(f"""
╔══════════════════════════════════════════════════╗
║   ÉVALUATION V3 — données réelles (gold)         ║
╠══════════════════════════════════════════════════╣
║  ARI   : {ari:+.4f}                              ║
║  NMI   : {nmi:.4f}                               ║
║  Msgs  : {len(pairs)} évalués                    ║
║  Gold  : {n_ep_gold} épisodes                    ║
║  Pred  : {n_ep_pred} épisodes                    ║
╚══════════════════════════════════════════════════╝
""")

if ari < 0.20:
    print("→ ARI faible")
    if n_ep_pred > n_ep_gold * 1.5:
        print("  Sur-fragmentation — réduire hard_break ou attach_threshold")
    elif n_ep_pred < n_ep_gold * 0.6:
        print("  Sous-segmentation — augmenter attach_threshold")
elif ari < 0.40:
    print("→ ARI moyen : paramètres à affiner")
else:
    print("→ ARI solide — bonne généralisation")

In [ ]:
# ── CELLULE 9 (optionnelle) : Grid search rapide ──────────────────────────
# Décommenter pour chercher les meilleurs paramètres

# import itertools
# results = []
# 
# for attach_thresh, hard_break in itertools.product(
#     [0.20, 0.25, 0.30, 0.35],
#     [360, 720, 1440, 0],
# ):
#     seg = EpisodeSegmenter(
#         time_threshold_minutes=120, attach_threshold=attach_thresh,
#         alpha=0.45, beta=0.25, gamma=0.10, delta=0.20, rho=0.05,
#         dormancy_minutes=1440, ema_alpha=0.80, active_penalty_hours=24.0,
#         hard_break_minutes=hard_break, allow_reactivation=True,
#     )
#     eps = seg.segment(artifacts, embeddings)
#     eps = seg.consolidate(eps)
#     yp = [None] * n
#     for ep in eps:
#         for idx in ep.artifact_indices:
#             if idx < n: yp[idx] = ep.id
#     pairs = [(t, p) for t, p in zip(y_true, yp) if t is not None and p is not None]
#     yt2, yp2 = zip(*pairs)
#     ari2 = adjusted_rand_score(yt2, yp2)
#     results.append({'attach': attach_thresh, 'hard_break': hard_break,
#                     'n_ep': len(eps), 'ari': ari2})
#     print(f"attach={attach_thresh} hard_break={hard_break:4d}  ARI={ari2:.4f}  eps={len(eps)}")
# 
# best = max(results, key=lambda x: x['ari'])
# print(f"\nMeilleur : {best}")